In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.transforms import v2
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import LeaveOneGroupOut
from torchinfo import summary
from scipy import signal
import pandas as pd
import numpy as np
import warnings
import math
import time
import os
warnings.filterwarnings('ignore')

C:\Users\abhin\AppData\Roaming\Python\Python312\site-packages\threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


In [2]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 150)

In [3]:
while not os.path.isdir(os.path.join(os.getcwd(), 'data')):
    os.chdir("../") # set cwd to root dir

**Enable CUDA if available**

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

device

NVIDIA GeForce RTX 3050 Ti Laptop GPU


device(type='cuda')

**Custom Transforms**

In [5]:
class ButterworthFilter(object):
    def __init__(self, cutoff, order, fs):
        self.cutoff = cutoff
        self.order = order
        self.fs = fs

    def __call__(self, input): # input must be numpy array
        nyquist = 0.5 * self.fs
        normal_cutoff = self.cutoff / nyquist
        b, a = signal.butter(self.order, normal_cutoff, btype='low', analog=False)
        smoothed_imu_signal = signal.filtfilt(b, a, input)

        return torch.from_numpy(smoothed_imu_signal.copy()) # convert back to tensor

In [6]:
class PadTrimToLength(object):
    def __init__(self, padlen):
        self.padlen = padlen

    def __call__(self, input):
        pad = nn.ZeroPad1d((0, max(0, self.padlen - input.shape[1])))(input)
        output = torch.narrow(pad, 1, 0, self.padlen)
        assert output.shape[1] == self.padlen, "Not equal to length of pad"
        return output

In [7]:
class Normalize(object):
    def __init__(self, mean, std, epsilon=1e-7):
        self.mean = mean
        self.std = std
        self.epsilon = epsilon

    def __call__(self, input):
        output = (input - self.mean) / (self.std + self.epsilon)
        return output

**Define dataset pipeline**

In [8]:
class DUO_GAIT(Dataset):
    def __init__(self, allowed_sensors, num_participants, ignore_participant_ids, remove_outliers=True, transform=None, target_transform=None):
        self.num_participants = num_participants
        self.allowed_sensors = allowed_sensors
        self.num_channels = len(allowed_sensors) * 6
        self.foot_strides_df = None
        self.sensor_dict = {}

        for participant_id in range(1, num_participants+1):
            if participant_id in ignore_participant_ids:
                continue

            for sensor_location in self.allowed_sensors:
                for protocol in ["control", "fatigue"]:
                    file_name = f"data/DUO-GAIT/interim/OG_st_{protocol}/sub_{participant_id:02}/{sensor_location}.csv"
                    sensor_df = pd.read_csv(file_name)
                    sensor_df.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)

                    self.sensor_dict[file_name] = sensor_df 
            # save interim files for future

            lf_control_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_control/sub_{participant_id:02}/left_foot_core_params.csv")
            lf_control_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
            lf_control_df['is_fatigue'] = 0
            lf_control_df['is_right_foot'] = 0
            lf_control_df['Participant'] = participant_id

            self.create_start_end_samples_strides(lf_control_df, participant_id, is_control=1)

            rf_control_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_control/sub_{participant_id:02}/right_foot_core_params.csv")
            rf_control_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
            rf_control_df['is_fatigue'] = 0
            rf_control_df['is_right_foot'] = 1
            rf_control_df['Participant'] = participant_id

            self.create_start_end_samples_strides(rf_control_df, participant_id, is_control=1)

            lf_fatigue_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_fatigue/sub_{participant_id:02}/left_foot_core_params.csv")
            lf_fatigue_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
            lf_fatigue_df['is_fatigue'] = 1
            lf_fatigue_df['is_right_foot'] = 0
            lf_fatigue_df['Participant'] = participant_id

            self.create_start_end_samples_strides(lf_fatigue_df, participant_id, is_control=0)

            rf_fatigue_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_fatigue/sub_{participant_id:02}/right_foot_core_params.csv")
            rf_fatigue_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
            rf_fatigue_df['is_fatigue'] = 1
            rf_fatigue_df['is_right_foot'] = 1
            rf_fatigue_df['Participant'] = participant_id

            self.create_start_end_samples_strides(rf_fatigue_df, participant_id, is_control=0)

            self.foot_strides_df = pd.concat([self.foot_strides_df, lf_control_df, rf_control_df, lf_fatigue_df, rf_fatigue_df], axis=0)

        if remove_outliers:
            self.foot_strides_df = self.foot_strides_df[self.foot_strides_df['is_outlier']==False].reset_index(drop=True)

        self.transform = transform
        self.target_transform = target_transform

    def create_start_end_samples_strides(self, df, participant_id, is_control):
        df.sort_values(by='stride_index', inplace=True)
        df['start_samples'] = df['ic_samples'].shift(1)
        target_time = df.loc[0, 'start_times']
        
        protocol = "control" if is_control else "fatigue"

        file_name = f"data/DUO-GAIT/interim/OG_st_{protocol}/sub_{participant_id:02}/{self.allowed_sensors[0]}.csv"        
        fatigue_df = self.sensor_dict[file_name].copy() # copy to avoid changing sensor dict's dataframes
        fatigue_df['Delta (secs)'] = fatigue_df['Time (secs)'] - fatigue_df['Time (secs)'].min() # delta time

        ts_eq_check = fatigue_df['Delta (secs)'].apply(lambda x: math.isclose(x, target_time, rel_tol=1e-5))
        start_sample = fatigue_df[ts_eq_check]['Sample'].item() - fatigue_df['Sample'].min()

        df.loc[0, 'start_samples'] = start_sample
        df['start_samples'] = df['start_samples'] + fatigue_df['Sample'].min()
        df['end_samples'] = df['ic_samples'] + fatigue_df['Sample'].min() - 1 # make it inclusive for ending samples too

        df['start_samples'] = df['start_samples'].astype(np.int64)
        df['end_samples'] = df['end_samples'].astype(np.int64)

    def __len__(self):
        return len(self.foot_strides_df)

    def __getitem__(self, idx):
        row = self.foot_strides_df.iloc[idx]

        imu_signals_df = pd.DataFrame()
        
        for sensor_location in self.allowed_sensors:
            protocol = "fatigue" if row['is_fatigue'] else "control"
            file_name = f"data/DUO-GAIT/interim/OG_st_{protocol}/sub_{row['Participant']:02}/{sensor_location}.csv"

            sensor_df = self.sensor_dict[file_name]
            sensor_df = sensor_df[(sensor_df['Sample'] >= row['start_samples']) & (sensor_df['Sample'] <= row['end_samples'])]

            for sensor_type in ["Gyr", "Acc"]:
                for direction in ["X", "Y", "Z"]:
                    col_name = f"{sensor_type}{direction}"
                    new_col_name = f"{sensor_location}_{col_name}"

                    imu_signals_df[new_col_name] = sensor_df[col_name]

        imu_signals_np = imu_signals_df.to_numpy().transpose().astype(np.float32)
        label = row["is_fatigue"]

        if self.transform:
            imu_signals_np = self.transform(imu_signals_np)

        if self.target_transform:
            label = self.target_transform(label)

        return imu_signals_np, label

**CNN Architecture**

In [9]:
class FatigueCNN(nn.Module):
    def __init__(self, in_channels, p_drop):
        super().__init__()

        self.features = nn.Sequential(
                        nn.Conv1d(in_channels=in_channels, out_channels=32, kernel_size=5, stride=1, padding=0),
                        nn.ReLU())

        self.gap = nn.AdaptiveMaxPool1d(output_size=1)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=p_drop)
        self.linear = nn.Linear(in_features=32, out_features=2)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.linear(x)
        
        return x

**Experiment Parameters**

In [10]:
num_participants = 18
ignore_participant_ids = [4,7,16]
allowed_sensors = ["LF", "RF", "SA"]
num_stride_samples = 150

**Hyper Parameters**

In [11]:
batch_size = 32
epochs = 5 # TODO - change this to infinite and implement early stopping instead
learning_rate = 0.02
min_learning_rate = 0.001
patience_epochs = 2
learning_rate_factor = 0.5
early_stopping_rounds = 3
train_percent = 0.90
p_drop = 0.5

**Leave one Participant out Training Loop**

In [12]:
print("Loading Dataset ...")
dataset = DUO_GAIT(allowed_sensors, num_participants, ignore_participant_ids)
print("Dataset Loaded ...")

dataset_size = len(dataset)
num_channels = dataset.num_channels
groups = dataset.foot_strides_df['Participant'].to_numpy()

gss = GroupShuffleSplit(n_splits=1, train_size=train_percent, random_state=42)
train_val_idx, test_idx = next(gss.split(X=dataset, groups=groups))

train_val_set = Subset(dataset, train_val_idx)
test_set = Subset(dataset, test_idx)

groups_train_val = groups[train_val_idx]

logo = LeaveOneGroupOut()
for i, (train_index, valid_index) in enumerate(logo.split(X=train_val_set, groups=groups_train_val)):
    train_set = Subset(train_val_set, train_index)
    valid_set = Subset(train_val_set, valid_index)

    dataset.transform = v2.Compose([
            ButterworthFilter(cutoff=10.0, order=3, fs=128.0),
            PadTrimToLength(padlen=num_stride_samples),
    ]) # default transform before normalization

    stat_loader = DataLoader(train_set, batch_size=batch_size)
    lopo = groups_train_val[valid_index][0]

    full_train_data = []
    for batch_idx, (train_features, train_labels) in enumerate(stat_loader):
        full_train_data.append(train_features)

    full_train_data = torch.concat(full_train_data, dim=0)
    std, mean = torch.std_mean(full_train_data, dim=(0, 2), keepdim=True)
    # compute mean and std over all channels separately

    std = torch.squeeze(std, dim=0)
    mean = torch.squeeze(mean, dim=0)
    # remove the batch dimension

    dataset.transform = v2.Compose([
        ButterworthFilter(cutoff=10.0, order=3, fs=128.0),
        PadTrimToLength(padlen=num_stride_samples),
        Normalize(mean=mean, std=std),
        v2.Lambda(lambda x: x.to(torch.float32))
    ]) # update transform for normalization

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=True)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, pin_memory=True)
    # create dataloaders for training and validation

    timestamp = time.strftime("%b-%d-%Y %I-%M-%S %p")
    writer = SummaryWriter(log_dir=f'runs/{timestamp}/lopo_{lopo:02} p_drop_{p_drop}', flush_secs=30)

    model = FatigueCNN(num_channels, p_drop).to(device=device) # move model to device

    loss = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=learning_rate_factor, patience=patience_epochs, min_lr=min_learning_rate)

    epochs_without_gain = 0
    best_valid_loss = float("inf")

    print(f"Beginning training for lopo_{lopo:02}")
    for epoch in range(0, epochs):
        train_epoch_loss, valid_epoch_loss = 0.0, 0.0

        train_epoch_preds, train_epoch_labels = [], []
        valid_epoch_preds, valid_epoch_labels = [], []

        train_start_time = time.time()
        model.train()
        for batch_idx, (train_features, train_labels) in enumerate(train_loader):
            train_batch_size = len(train_labels)

            train_features = train_features.to(device)
            train_labels = train_labels.to(device) # move data to device

            optimizer.zero_grad()

            logits = model(train_features)
            predictions = torch.argmax(logits, dim=1)
            
            train_epoch_preds.extend(predictions.cpu().numpy())
            train_epoch_labels.extend(train_labels.cpu().numpy())

            train_batch_loss = loss(logits, train_labels)
            train_batch_loss.backward()

            optimizer.step()

            train_epoch_loss += train_batch_loss.item() * train_batch_size
        train_end_time = time.time()

        model.eval()
        valid_start_time = time.time()
        with torch.no_grad():
            for batch_idx, (valid_features, valid_labels) in enumerate(valid_loader):
                valid_batch_size = len(valid_labels)
                
                valid_features = valid_features.to(device)
                valid_labels = valid_labels.to(device) # move to device
                
                logits = model(valid_features)
                predictions = torch.argmax(logits, dim=1)

                valid_epoch_preds.extend(predictions.cpu().numpy())
                valid_epoch_labels.extend(valid_labels.cpu().numpy())

                valid_batch_loss = loss(logits, valid_labels)
                valid_epoch_loss += valid_batch_loss.item() * valid_batch_size
        valid_end_time = time.time()

        train_epoch_loss /= len(train_set)
        valid_epoch_loss /= len(valid_set)

        scheduler.step(valid_epoch_loss) # step the scheduler and reduce lr if necessary

        train_epoch_acc = accuracy_score(train_epoch_labels, train_epoch_preds)
        valid_epoch_acc = accuracy_score(valid_epoch_labels, valid_epoch_preds)

        writer.add_scalar("Loss/train-epoch", train_epoch_loss, epoch)
        writer.add_scalar('Accuracy/train-epoch', train_epoch_acc, epoch)
        writer.add_scalar('Elapsed Time/train-epoch-secs', train_end_time - train_start_time, epoch)

        writer.add_scalar("Loss/valid-epoch", valid_epoch_loss, epoch)
        writer.add_scalar('Accuracy/valid-epoch', valid_epoch_acc, epoch)
        writer.add_scalar('Elapsed Time/valid-epoch-secs', valid_end_time - valid_start_time, epoch)

        output_dir = f"checkpoints/lopo_{lopo:02}"
        os.makedirs(output_dir, exist_ok=True) # create checkpoints dirs

        torch.save(model.state_dict(), f"{output_dir}/epoch_{epoch+1}.pth") # checkpoint for safety

        if valid_epoch_loss < best_valid_loss:
            best_valid_loss = valid_epoch_loss
            epochs_without_gain = 0
        else:
            epochs_without_gain += 1

        if epochs_without_gain >= early_stopping_rounds:
            break

Loading Dataset ...
Dataset Loaded ...
Beginning training for lopo_01
Beginning training for lopo_02
Beginning training for lopo_03
Beginning training for lopo_05
Beginning training for lopo_06
Beginning training for lopo_08
Beginning training for lopo_09
Beginning training for lopo_10
Beginning training for lopo_11
Beginning training for lopo_13
Beginning training for lopo_15
Beginning training for lopo_17
Beginning training for lopo_18
